# AstraVault ML Experiment

Exploratory workflow for mission success prediction, rocket reliability scoring, and failure pattern analysis. The notebook uses the same feature families as the FastAPI service so experiments can graduate cleanly into `train.py` and `predict.py`.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ModuleNotFoundError:
    plt = None
    sns = None

if sns is not None:
    sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

DATA_PATH = Path("../data/missions.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/missions.csv")

RANDOM_STATE = 42

## 2. Load and Enrich Mission Data

In [ ]:
missions = pd.read_csv(DATA_PATH)

missions["rocket_reliability"] = (
    (missions["successful_launches"] + missions["partial_failures"] * 0.45)
    / missions["total_launches"].replace(0, np.nan)
).fillna(missions["launch_vehicle_history"].clip(0, 1))
missions["launch_site_history"] = missions["launch_site_success_rate"].clip(0, 1)
missions["success_label"] = (missions["status"] == "SUCCESS").astype(int)
missions["crewed"] = missions["crewed"].astype(int)

missions.head()

In [ ]:
summary = pd.DataFrame(
    {
        "rows": [len(missions)],
        "missions": [missions["mission_name"].nunique()],
        "rockets": [missions["rocket_name"].nunique()],
        "organizations": [missions["organization"].nunique()],
        "success_rate": [missions["success_label"].mean().round(3)],
    }
)
summary

## 3. Exploratory Checks

In [ ]:
outcome_snapshot = {
    "status_counts": missions["status"].value_counts(),
    "orbit_counts": missions["orbit_type"].value_counts(),
    "payload_by_status": missions.groupby("status")["payload_mass_kg"].describe().round(2),
}

if plt is None or sns is None:
    outcome_snapshot
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    outcome_snapshot["status_counts"].plot(kind="bar", ax=axes[0], color="#33658a")
    axes[0].set_title("Mission Outcomes")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Missions")

    outcome_snapshot["orbit_counts"].plot(kind="bar", ax=axes[1], color="#55a630")
    axes[1].set_title("Orbit Mix")
    axes[1].set_xlabel("")

    sns.boxplot(data=missions, x="status", y="payload_mass_kg", ax=axes[2], color="#f6ae2d")
    axes[2].set_title("Payload Mass by Outcome")
    axes[2].set_xlabel("")
    axes[2].set_ylabel("Payload mass kg")

    plt.tight_layout()

In [ ]:
missions.groupby(["organization", "status"]).size().unstack(fill_value=0).sort_index()

## 4. Mission Success Model

In [ ]:
NUMERIC_FEATURES = [
    "rocket_reliability",
    "organization_experience",
    "launch_vehicle_history",
    "payload_mass_kg",
    "previous_failures",
    "launch_site_history",
    "crewed",
    "total_launches",
    "successful_launches",
    "failed_launches",
    "partial_failures",
]

CATEGORICAL_FEATURES = [
    "orbit_type",
    "mission_type",
    "destination",
    "budget_level",
    "reliability_priority",
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
target = "success_label"

In [ ]:
x = missions[FEATURES]
y = missions[target]
stratify = y if y.nunique() > 1 and y.value_counts().min() > 1 else None

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=stratify,
)

mission_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

mission_model = Pipeline(
    steps=[
        ("preprocess", mission_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=240,
                max_depth=7,
                min_samples_leaf=1,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

mission_model.fit(x_train, y_train)
y_pred = mission_model.predict(x_test)

metrics = pd.DataFrame(
    [
        {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
        }
    ]
).round(4)
metrics

In [ ]:
feature_names = mission_model.named_steps["preprocess"].get_feature_names_out()
importances = mission_model.named_steps["model"].feature_importances_

importance_frame = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(12)
)

if plt is not None and sns is not None:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=importance_frame, x="importance", y="feature", color="#33658a")
    plt.title("Top Mission Success Features")
    plt.xlabel("Random forest importance")
    plt.ylabel("")
    plt.tight_layout()

importance_frame

## 5. Rocket Reliability Model

In [ ]:
rocket_features = [
    "rocket_name",
    "total_launches",
    "successful_launches",
    "failed_launches",
    "partial_failures",
    "organization_experience",
    "launch_vehicle_history",
]

rocket_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            [
                "total_launches",
                "successful_launches",
                "failed_launches",
                "partial_failures",
                "organization_experience",
                "launch_vehicle_history",
            ],
        ),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), ["rocket_name"]),
    ]
)

rocket_model = Pipeline(
    steps=[
        ("preprocess", rocket_preprocessor),
        ("model", RandomForestRegressor(n_estimators=160, max_depth=6, random_state=RANDOM_STATE)),
    ]
)

rocket_model.fit(missions[rocket_features], missions["rocket_reliability"].clip(0, 1))

rocket_scores = (
    missions.assign(predicted_reliability=rocket_model.predict(missions[rocket_features]))
    .groupby("rocket_name")
    .agg(
        missions=("mission_name", "count"),
        total_launches=("total_launches", "max"),
        observed_reliability=("rocket_reliability", "max"),
        predicted_reliability=("predicted_reliability", "mean"),
    )
    .sort_values(["predicted_reliability", "total_launches"], ascending=False)
)

rocket_scores.head(10).round(4)

## 6. Failure Pattern Review

In [ ]:
failures = missions[missions["status"].isin(["FAILURE", "PARTIAL"])].copy()
failures["failure_category"] = failures["failure_category"].fillna("unknown")

failure_patterns = (
    failures.groupby("failure_category")
    .agg(frequency=("mission_name", "count"), examples=("mission_name", lambda values: ", ".join(values.head(3))))
    .assign(percentage=lambda frame: (frame["frequency"] / max(frame["frequency"].sum(), 1) * 100).round(2))
    .sort_values("frequency", ascending=False)
)

failure_patterns

## 7. Scenario Prediction

In [ ]:
def risk_level(probability: float) -> str:
    if probability >= 0.82:
        return "LOW"
    if probability >= 0.62:
        return "MEDIUM"
    if probability >= 0.38:
        return "HIGH"
    return "CRITICAL"


scenario = pd.DataFrame(
    [
        {
            "rocket_reliability": 0.92,
            "organization_experience": 0.86,
            "launch_vehicle_history": 0.9,
            "payload_mass_kg": 5200,
            "previous_failures": 0,
            "launch_site_history": 0.91,
            "crewed": 0,
            "total_launches": 120,
            "successful_launches": 113,
            "failed_launches": 2,
            "partial_failures": 5,
            "orbit_type": "LEO",
            "mission_type": "SATELLITE_DEPLOYMENT",
            "destination": "Low Earth Orbit",
            "budget_level": "MEDIUM",
            "reliability_priority": "HIGH",
        }
    ],
    columns=FEATURES,
)

success_probability = mission_model.predict_proba(scenario)[0][1]
pd.DataFrame(
    [
        {
            "success_probability": round(success_probability, 4),
            "risk_level": risk_level(success_probability),
            "confidence_score": round(min(0.97, 0.58 + abs(success_probability - 0.5) * 0.78), 4),
        }
    ]
)

## 8. Notes for Promotion

- Keep feature names aligned with `train.py` before moving notebook experiments into service code.
- Re-run the notebook after expanding `data/missions.csv`; the current dataset is intentionally small, so metrics are directional rather than production-grade.
- Use the trained service artifacts in `ml-service/models/` for API-backed pages once the model behavior is accepted.